## Kafka

### Offsets

- In Kafka, every message has a unique ID number called an Offset. 

- As your flink app reads messages, it marks them as "Read" by committing that ID back to Kafka

### Redeploying flink

- Flink is stateful processing of streaming data

- That means that a flink app doesn't just "process" stuff, it also remembers 
    - What point was the data processed up to? (i.e. which offset, see section on offsets)
    - What was the state of the last processed data?

- How does flink do this?
    - For checkpoints, it periodically takes a snapshot and persists it to some blob storage for later use
    - For savepoints, user will trigger a snapshot to be persisted to some blob storage 

- Checkpoint vs Savepoint
    - Flink allows you to do both checkpointing and savepointing
    - Checkpointing happens automatically; the app periodically takes a snapshot of the state, and the point that it last processed
    - Savepoints are **manually** triggered checkpoints. Basically you tell flink you want to do something, so please save and persist the snapshot to s3 for resumption down the line


- With this background in mind, let's talk through the deployment strategies:
    - Deploy with Checkpoints Invalidated: 
        - "Clean slate" deployment
        - Ignore the data that may be missed between now and the last checkpoint. Just go ahead and process from the latest available data
        - If you are processing a 60min window, you will need to accumulate for 60 mins before you get a value
    
    - Restart from Checkpoint
        - Look for the last automatic backup and resume from there
        - You MIGHT miss a few minutes of data (depending on how long the app was down), but ideally the window won't be empty
    
    - Restart from Provided Savepoint
        - Manual restore
        - Restart from my last defined point. Treat it as a "rollback" to some snapshotted state
    
    - Take Savepoint and Deploy with Checkpoints Invalidated 
        - Gracefully stops the app, and takes a savepoint
        - But deploy as if you don't care about the savepoint
        - This is mostly useful "just in case" you realise you needed the savepoint data
    
    - Take Savepoint and Restart from Checkpoint
        - Take a "snapshot" for future reference, but restart from checkpoint instead
    
    - Take Savepoint and Restart from Provided Savepoint
        - Take a snapshot for reference, but restart from another snapshot

    - (Default) Take Savepoint and Restart from Savepoint
        - Take a snapshot for reference, and restart from that snapshot